Задание 1

Напишите функцию, которая возвращает название валюты (поле ‘Name’) с максимальным значением курса с помощью сервиса 
(https://www.cbr-xml-daily.ru/daily_json.js)

In [18]:
import sys
sys.path.insert(0, r'C:\Users\yaakt\Desktop\Netology_Python\.venv\Lib\site-packages')

In [19]:
import requests

def get_currency_with_max_rate():
    try:
        # Загружаем данные с API Центрального банка
        response = requests.get('https://www.cbr-xml-daily.ru/daily_json.js')
        response.raise_for_status()  # Проверяем успешность запроса
        
        # Преобразуем JSON в словарь Python
        data = response.json()
        
        # Получаем словарь валют
        currencies = data.get('Valute', {})
        
        if not currencies:
            return "Не удалось получить данные о валютах"
        
        # Находим валюту с максимальным курсом
        max_currency = max(currencies.items(), 
                          key=lambda x: x[1]['Value'] / x[1]['Nominal'])
        
        # Возвращаем название валюты
        return max_currency[1]['Name']
    
    except requests.exceptions.RequestException as e:
        return f"Ошибка при получении данных: {e}"
    except Exception as e:
        return f"Произошла ошибка: {e}"

# Пример использования
result = get_currency_with_max_rate()
print(f"Валюта с максимальным курсом: {result}")

Валюта с максимальным курсом: Бахрейнский динар


Задание 2

Добавьте в класс Rate параметр diff (со значениями True или False), который в случае значения True в методах курсов валют (eur, usd итд) будет возвращать не курс валюты, а изменение по сравнению в прошлым значением. Считайте, self.diff будет принимать значение True только при возврате значения курса. При отображении всей информации о валюте он не используется.

In [20]:
import requests

class Rate:
    def __init__(self, diff=False):
        self.format = 'value'
        self.diff = diff
        self.data = self._get_data()
    
    def _get_data(self):
        try:
            response = requests.get('https://www.cbr-xml-daily.ru/daily_json.js')
            return response.json()
        except Exception as e:
            print(f"Ошибка: {e}")
            return {'Valute': {}}
    
    def _get_rate(self, currency_code):
        if 'Valute' not in self.data:
            return None
        
        currency = self.data['Valute'].get(currency_code)
        if not currency:
            return None
        
        if self.diff:
            return round(currency['Value'] - currency['Previous'], 4)
        else:
            return currency['Value']
    
    @property
    def usd(self):
        return self._get_rate('USD')
    
    @property
    def eur(self):
        return self._get_rate('EUR')
    
    def get_currency_info(self, currency_code):
        if 'Valute' in self.data:
            return self.data['Valute'].get(currency_code, {})
        return {}


rate = Rate(diff=True)
usd_info = rate.get_currency_info('USD')
eur_info = rate.get_currency_info('EUR')

print("Информация о USD:", usd_info)
print("Информация о EUR:", eur_info)
print(f"Изменение USD: {rate.usd}")  
print(f"Изменение EUR: {rate.eur}")

Информация о USD: {'ID': 'R01235', 'NumCode': '840', 'CharCode': 'USD', 'Nominal': 1, 'Name': 'Доллар США', 'Value': 75.9246, 'Previous': 76.0382}
Информация о EUR: {'ID': 'R01239', 'NumCode': '978', 'CharCode': 'EUR', 'Nominal': 1, 'Name': 'Евро', 'Value': 89.0589, 'Previous': 88.7898}
Изменение USD: -0.1136
Изменение EUR: 0.2691


Задание 3

Напишите класс Designer, который учитывает количество международных премий. Подсказки в коде занятия (“Ноутбук к лекциям «Понятие класса» + презентация”, zip-файл “Используемый ноутбук к лекциям «Понятие класса»).

Дизайнеры
Повышение на 1 грейд за каждые 7 баллов. Получение международной
премии – это +2 балла

In [21]:
class Employee:
    def __init__(self, name, seniority):
        self.name = name
        self.seniority = seniority
        
        self.grade = 1
    
    def grade_up(self):
        """Повышает уровень сотрудника"""
        self.grade += 1
    
    def publish_grade(self):
        """Публикация результатов аккредитации сотрудников"""
        print(self.name, self.grade)
    
    def check_if_it_is_time_for_upgrade(self):
        pass


In [22]:
class Designer(Employee):
    def __init__(self, name, seniority, awards=0):
        super().__init__(name, seniority)
        # awards — количество международных премий
        self.awards = awards
        # баллы = 2 за каждую премию
        self.points = awards * 2

    def check_if_it_is_time_for_upgrade(self):
        self.seniority += 1

        # повышение грейда за каждые 7 баллов
        if self.points >= 7:
            upgrades = self.points // 7
            for _ in range(upgrades):
                self.grade_up()
            self.points %= 7

        return self.publish_grade()

    def add_award(self):
        """Добавить международную премию дизайнеру (+2 балла)"""
        self.awards += 1
        self.points += 2
        # проверим, не настало ли время повышения
        self.check_if_it_is_time_for_upgrade()

In [23]:
d = Designer("Maria Shmatko", seniority=0, awards=0)

d.check_if_it_is_time_for_upgrade()  
d.add_award()                       
d.add_award()                        
d.add_award()                        
d.add_award()                        
d.add_award()                        
d.add_award() 

Maria Shmatko 1
Maria Shmatko 1
Maria Shmatko 1
Maria Shmatko 1
Maria Shmatko 2
Maria Shmatko 2
Maria Shmatko 2
